# 04 - Build PQENS087 Prequential Datasets

            This notebook expands the frozen common master quarter by quarter. Each evaluation quarter trains only on the preceding 40 target quarters, adds time-decay weights and produces exactly one score row for each evaluated bank-quarter.


In [ ]:
from pathlib import Path

def find_package_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "01_DATA_PIPELINE").is_dir() and (candidate / "02_MODELS").is_dir():
            return candidate
    if current.name == "01_DATA_PIPELINE":
        return current.parent
    raise RuntimeError("Open this notebook from inside the SUBMIT_THIS package.")

PACKAGE_ROOT = find_package_root()
PIPELINE_DIR = PACKAGE_ROOT / "01_DATA_PIPELINE"
FROZEN_RAW_DIR = PIPELINE_DIR / "FROZEN_RAW_DATA"
GENERATED_DIR = PIPELINE_DIR / "GENERATED_OUTPUTS"
GENERATED_DIR.mkdir(parents=True, exist_ok=True)
print("Package root:", PACKAGE_ROOT)


In [ ]:
import json
import numpy as np
import pandas as pd

DATA_SOURCE = "frozen"  # Change to "generated" after notebook 03 finishes.
if DATA_SOURCE == "frozen":
    MASTER_PATH = PACKAGE_ROOT / "02_MODELS" / "04_ENS035" / "common_master.csv"
else:
    MASTER_PATH = GENERATED_DIR / "03_NEW_MASTER" / "exports" / "common_master.csv"

OUTPUT_DIR = GENERATED_DIR / "04_PQ_PREQUENTIAL"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FROZEN_TEST = PACKAGE_ROOT / "02_MODELS" / "05_PQENS087" / "pq087_prequential.csv"
FROZEN_VALIDATION = PACKAGE_ROOT / "02_MODELS" / "05_PQENS087" / "pq087_validation_prequential.csv"

master = pd.read_csv(MASTER_PATH, encoding="utf-8-sig")
metadata = [
    "row_key", "ticker", "report_year", "report_quarter", "quarter_index",
    "feature_quarter", "target_year", "target_quarter_number", "target_q_index",
    "target_quarter", "risk10_numeric", "risk10_label", "aux_risk5_numeric",
    "aux_risk5_label", "mg002_distance_weight", "mg035_positive_weight",
]
features = [column for column in master.columns if column not in metadata]
assert len(features) == 357, len(features)
print("Master:", MASTER_PATH)
print("Rows / regular features:", len(master), len(features))


In [ ]:
def build_prequential(frame: pd.DataFrame, target_indices: list[int]) -> pd.DataFrame:
    parts = []
    keep = metadata + features
    for target_index in sorted(int(value) for value in target_indices):
        evaluation = frame[frame.target_q_index == target_index].copy()
        train = frame[
            (frame.target_q_index < target_index)
            & (frame.target_q_index >= target_index - 40)
        ].copy()
        age_years = (target_index - train.target_q_index) / 4.0
        train["pq_sample_weight"] = np.power(0.95, age_years)
        train["pq_age_years"] = age_years
        train["pq_role"] = "train"
        evaluation["pq_sample_weight"] = 1.0
        evaluation["pq_age_years"] = 0.0
        evaluation["pq_role"] = "score"
        evaluation_quarter = str(evaluation.target_quarter.iloc[0])
        for part in (train, evaluation):
            part["evaluation_target_q_index"] = target_index
            part["evaluation_target_quarter"] = evaluation_quarter
            parts.append(part[[
                "evaluation_target_q_index", "evaluation_target_quarter", "pq_role",
                "pq_sample_weight", "pq_age_years", *keep,
            ]])
    return pd.concat(parts, ignore_index=True)

test_indices = master.loc[master.target_year >= 2024, "target_q_index"].unique().tolist()
validation_indices = master.loc[master.target_year.between(2020, 2023), "target_q_index"].unique().tolist()

test_pq = build_prequential(master, test_indices)
validation_pq = build_prequential(master, validation_indices)

test_path = OUTPUT_DIR / "pq087_prequential.csv"
validation_path = OUTPUT_DIR / "pq087_validation_prequential.csv"
test_pq.to_csv(test_path, index=False, encoding="utf-8-sig")
validation_pq.to_csv(validation_path, index=False, encoding="utf-8-sig")


In [ ]:
checks = [
    {"file": "pq087_prequential.csv", "rows": len(test_pq), "columns": test_pq.shape[1],
     "score_rows": int((test_pq.pq_role == "score").sum()),
     "unique_score_keys": test_pq.loc[test_pq.pq_role == "score", "row_key"].nunique()},
    {"file": "pq087_validation_prequential.csv", "rows": len(validation_pq), "columns": validation_pq.shape[1],
     "score_rows": int((validation_pq.pq_role == "score").sum()),
     "unique_score_keys": validation_pq.loc[validation_pq.pq_role == "score", "row_key"].nunique()},
]
checks_frame = pd.DataFrame(checks)
display(checks_frame)

assert checks[0]["rows"] == 8234 and checks[0]["columns"] == 378
assert checks[0]["score_rows"] == checks[0]["unique_score_keys"] == 234
assert checks[1]["rows"] == 11306 and checks[1]["columns"] == 378
assert checks[1]["score_rows"] == checks[1]["unique_score_keys"] == 416

(OUTPUT_DIR / "NOTEBOOK_ACCEPTANCE.json").write_text(
    json.dumps(checks, indent=2), encoding="utf-8"
)
print("PQ prequential exports completed:", OUTPUT_DIR)
